### Étape 1 — Distribution de target Purchased

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style="whitegrid")

# le chemin de fichier de train_final.csv
train_path = '../data/train_final.csv'
df_train = pd.read_csv(train_path)

print(f"Shape de Train data: {df_train.shape}")
print(f"Les colonnes: {df_train.columns.tolist()}")
df_train.head()

In [ ]:
print("=== Distribution de Purchased ===")
# combien de sessions ont été achetées (Purchased = 1) et non achetées (Purchased = 0)
counts = df_train['Purchased'].value_counts()
# proportion de chaque classe (0/1)
percentages = df_train['Purchased'].value_counts(normalize=True) * 100

# Affichage dans Console
# val = variable dans Purchased qui prend les valeurs 0 et 1
for val in counts.index:
    label = "Achat (1)" if val == 1 else "Non Achat (0)"
    # le nombre de sessions ayant meme classe counts[val] et son pourcentage percentages[val]
    print(f"{label}: {counts[val]} sessions ({percentages[val]:.2f}%)")

# 2. Visualisation (Comptage + Pourcentage)
# une figure avec 2 sous-graphes (1 ligne, 2 colonnes)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Graph 1: Countplot geres par axes[0] 
# hue = 'Purchased' pour colorer les barres selon la classe (0 ou 1)
sns.countplot(data=df_train, x='Purchased', hue='Purchased',  palette=['#ff9999', '#66b3ff'], ax=axes[0])
axes[0].set_title('Counts of Purchased (0 vs 1)', fontsize=14)
axes[0].set_xlabel('Purchased (0 = Non Achat, 1 = Achat)')
axes[0].set_ylabel('Count of Sessions')

# patches = les barres du countplot, on va annoter chaque barre avec son nombre de sessions
for p in axes[0].patches:
    #p.get_height() = hauteur de la barre (nombre de sessions), p.get_x() = position x de la barre, p.get_width() = largeur de la barre
    axes[0].annotate(f'{int(p.get_height())}', 
                     (p.get_x() + p.get_width() / 2., p.get_height()), 
                     ha='center', va='center', xytext=(0, 5), 
                     textcoords='offset points', fontsize=12)

# Graph 2: Pie Chart (Pourcentage)
axes[1].pie(counts, labels=['Non Achat (0)', 'Achat (1)'], 
            autopct='%1.2f%%', startangle=90, 
            colors=['#ff9999', '#66b3ff'], explode=(0, 0.1), shadow=True)
axes[1].set_title('Pourcentage de Purchased', fontsize=14)

plt.tight_layout()
plt.show()

# 3. La Décision (Conclusion)
print("\n=== La Décision ===")
if percentages[0] > 60:
    print(" Data Imbalanced!")
    print(" il faut utiliser'class_weight=balanced' .")
else:
    print("  Data Balanced ")

### Conclusion — Déséquilibre des classes

L'analyse de la variable cible `Purchased` montre un déséquilibre entre les deux classes. La classe 0 (Non Achat) est majoritaire, tandis que la classe 1 (Achat) est minoritaire.

Ce déséquilibre peut influencer l'apprentissage du modèle et le rendre davantage orienté vers la classe majoritaire.

Pour traiter ce problème, nous allons commencer par tester la régression logistique avec `class_weight='balanced'`, puis comparer ses performances avec un modèle sans pondération.

L'évaluation sera basée principalement sur l'Average Precision (AP), ainsi que sur le Recall, la Precision et le F1-score.

 ### Étape 2 — Distributions des variables numériques
 Pour comprendre comment les valeurs de chaque variable sont réparties. 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df_train = pd.read_csv('../data/train_final.csv')

# Les variables numériques à étudier
numeric_cols = [
    'AdminPages',
    'AdminTimeSec',
    'InfoPages',
    'InfoTimeSec',
    'CatalogPages',
    'CatalogTimeSec',
    'BounceRate',
    'ExitRate',
    'EngagementValue',
    'PromoProximity'
]

# Création des histogrammes
# definir la taille de la figure 
plt.figure(figsize=(16, 12))

for i, col in enumerate(numeric_cols):
    # 4 lignes et 3 colonnes de sous-graphes, i + 1 = position du sous-graphe
    plt.subplot(4, 3, i + 1)
    # bins = divise les valeurs en 30 intervalles , kde = True pour ajouter la courbe de densité
    sns.histplot(data=df_train, x=col, bins=30, kde=True)
    plt.title(f'Distribution de {col}')
    plt.xlabel(col)
    plt.ylabel('Fréquence')

plt.tight_layout()
plt.show()

1. AdminPages :

Observation :
- La majorité des sessions ont un nombre faible de pages administratives visitées.
- La fréquence diminue lorsque le nombre de pages augmente.
- Quelques sessions présentent des valeurs très élevées, jusqu'à environ 27 pages.

Interprétation :

La distribution de AdminPages est asymétrique à droite. Cela signifie que la plupart des visiteurs consultent peu de pages administratives, tandis qu'un petit nombre de visiteurs en consultent beaucoup.

2. AdminTimeSec

Observation :
- La majorité des sessions ont un temps de consultation très faible.
- La fréquence diminue rapidement lorsque le temps augmente.
- Quelques sessions présentent des durées très élevées, jusqu'à environ 3 500 secondes.

Interprétation :

La distribution de AdminTimeSec est fortement asymétrique à droite. La plupart des visiteurs passent peu de temps sur les pages administratives, alors que quelques visiteurs y passent beaucoup plus de temps.

3. InfoPages

Observation :
- La majorité des sessions ont un nombre très faible de pages informatives visitées.
- La fréquence diminue lorsque le nombre de pages augmente.
- Quelques sessions présentent un nombre élevé de pages visitées.

Interprétation :

La distribution de InfoPages est asymétrique à droite. La plupart des visiteurs consultent peu de pages informatives, tandis qu'un petit nombre en consultent davantage.
4. InfoTimeSec

Observation :
- La majorité des sessions présentent un temps de consultation faible.
- Les valeurs élevées sont beaucoup moins fréquentes.
- Quelques sessions présentent des durées très importantes, jusqu'à environ 2 500 secondes.

Interprétation :

La distribution de InfoTimeSec est fortement asymétrique à droite. La plupart des visiteurs passent peu de temps sur les pages informatives, tandis que quelques-uns y passent beaucoup plus de temps.
5. CatalogPages

Observation :
- La majorité des sessions présentent un nombre faible de pages de catalogue visitées.
- La fréquence diminue progressivement lorsque le nombre de pages augmente.
- Quelques sessions présentent des valeurs très élevées, jusqu'à environ 650 pages.

Interprétation :

La distribution de CatalogPages est asymétrique à droite. La plupart des visiteurs consultent un nombre limité de pages du catalogue, tandis qu'un petit nombre en consultent beaucoup.
6. CatalogTimeSec

Observation :
- La majorité des sessions ont un temps de consultation faible.
- La fréquence diminue rapidement lorsque le temps augmente.
- Quelques sessions présentent des durées très élevées, dépassant 40 000 secondes.

Interprétation :

La distribution de CatalogTimeSec est fortement asymétrique à droite. La plupart des visiteurs passent peu de temps à consulter le catalogue, tandis que quelques sessions présentent des durées beaucoup plus importantes.
7. BounceRate

Observation :
- La majorité des valeurs sont proches de zéro.
- La fréquence diminue lorsque le taux augmente.
- Quelques valeurs atteignent environ 0,20.

Interprétation :

La distribution de BounceRate est asymétrique à droite. La plupart des sessions présentent un taux de rebond faible, tandis qu'un petit nombre présentent un taux plus élevé.
8. ExitRate

Observation :
- Les valeurs sont principalement concentrées entre 0,01 et 0,05.
- La fréquence diminue progressivement lorsque le taux augmente.
- On observe également une concentration de valeurs autour de 0,20.

Interprétation :

La distribution de ExitRate est asymétrique à droite. La majorité des sessions présentent un taux de sortie relativement faible, mais certaines présentent un taux plus élevé.
9. EngagementValue

Observation :
- La majorité des valeurs sont proches de zéro.
- La fréquence diminue rapidement lorsque la valeur augmente.
- Quelques sessions présentent des valeurs très élevées, jusqu'à environ 180.

Interprétation :

La distribution de EngagementValue est fortement asymétrique à droite. La plupart des sessions ont une faible valeur d'engagement, tandis que quelques sessions présentent un engagement beaucoup plus important.
10. PromoProximity

Observation :
- La majorité des valeurs sont égales ou proches de zéro.
- On observe quelques concentrations autour de 0,2 ; 0,4 ; 0,6 ; 0,8 et 1.
- Les valeurs élevées sont beaucoup moins fréquentes.

Interprétation :

La distribution de PromoProximity est concentrée principalement autour de zéro. Cela signifie que la majorité des sessions présentent une faible proximité avec une promotion, tandis que les autres valeurs sont moins fréquentes.

### Conclusion de l'analyse des distributions numériques

L'analyse des distributions numériques montre que la majorité des variables présentent une asymétrie à droite. Les valeurs sont généralement concentrées autour des faibles valeurs, avec quelques observations très élevées.

Les variables `AdminPages`, `AdminTimeSec`, `InfoPages`, `InfoTimeSec`, `CatalogPages`, `CatalogTimeSec` et `EngagementValue` présentent notamment des valeurs extrêmes.

Les variables `BounceRate` et `ExitRate` sont également concentrées autour de faibles valeurs. Enfin, `PromoProximity` présente une forte concentration autour de zéro.

Ces observations nous permettent de mieux comprendre la structure des données et d'identifier les variables qui nécessitent une attention particulière lors de la préparation des données et de la construction des modèles.

### Étape 3 — Corrélations entre les variables numériques et Purchased

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Charger les données
df_train = pd.read_csv('../data/train_final.csv')

# Les variables numériques à étudier
numeric_cols = [
    'AdminPages', 'AdminTimeSec',
    'InfoPages', 'InfoTimeSec',
    'CatalogPages', 'CatalogTimeSec',
    'BounceRate', 'ExitRate',
    'EngagementValue', 'PromoProximity'
]

# Calculer la corrélation avec Purchased
correlations = df_train[numeric_cols + ['Purchased']].corr()

# Extraire les corrélations avec Purchased
corr_purchased = correlations[['Purchased']].drop('Purchased')

# Trier les corrélations par ordre décroissant
corr_purchased = corr_purchased.sort_values(
    by='Purchased', ascending=False
)

# Afficher les résultats
print("Corrélation des variables numériques avec Purchased :")
print(corr_purchased)

# Visualisation des corrélations avec Purchased
plt.figure(figsize=(6, 8))

sns.heatmap(
    corr_purchased,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1
)

plt.title("Corrélation des variables numériques avec Purchased")
plt.tight_layout()
plt.show()


Interprétation de chaque variable

1. EngagementValue : +0,339 (Corrélation positive modérée)

Interprétation : Les sessions ayant une valeur d'engagement plus élevée ont tendance à être davantage associées à un achat.

2. CatalogPages : +0,201 (Corrélation positive faible)

Interprétation : Les visiteurs qui consultent davantage de pages du catalogue ont légèrement plus tendance à effectuer un achat.

3. CatalogTimeSec : +0,181 (Corrélation positive faible)
 
Interprétation : Un temps de consultation plus élevé du catalogue est associé à une légère augmentation de la tendance à acheter.

4. AdminPages : +0,103 (Corrélation positive très faible)

Interprétation : Le nombre de pages administratives consultées présente une relation positive très faible avec l'achat.

5. AdminTimeSec : +0,100 (Corrélation positive très faible)

Interprétation : Le temps passé sur les pages administratives présente une relation positive très faible avec Purchased.

6. InfoPages : +0,050 (Corrélation positive très faible)

Interprétation : Le nombre de pages informatives consultées présente une relation linéaire presque inexistante avec l'achat.

7. InfoTimeSec : +0,039 (Corrélation positive très faible)

Interprétation : Le temps passé sur les pages informatives présente également une relation linéaire presque inexistante avec l'achat.

8. PromoProximity : +0,009 (Corrélation presque nulle)

Interprétation : Cette variable ne présente pratiquement aucune relation linéaire avec Purchased.

9. BounceRate : -0,132 (Corrélation négative faible)

Interprétation : Lorsque le taux de rebond augmente, la tendance à effectuer un achat diminue légèrement.

10. ExitRate : -0,188 (Corrélation négative faible) 

Interprétation : Lorsque le taux de sortie augmente, la tendance à effectuer un achat diminue légèrement.

### Interprétation des corrélations avec Purchased

L'analyse des corrélations montre que `EngagementValue` présente la relation positive la plus importante avec `Purchased`, avec un coefficient de 0,339. Cela indique que les sessions ayant un niveau d'engagement élevé sont généralement davantage associées à un achat.

Les variables `CatalogPages` (0,201) et `CatalogTimeSec` (0,181) présentent également des corrélations positives faibles. Cela suggère que la consultation du catalogue peut être liée à la décision d'achat.

En revanche, `BounceRate` (-0,132) et `ExitRate` (-0,188) présentent des corrélations négatives faibles. Une augmentation de ces taux est associée à une tendance plus faible à effectuer un achat.

Les autres variables présentent des corrélations très faibles ou presque nulles avec `Purchased`.

En conclusion, `EngagementValue`, `CatalogPages` et `CatalogTimeSec` semblent être les variables numériques les plus intéressantes pour expliquer la tendance à l'achat , `ExitRate`et `BounceRate` présentent des relations négatives avec l'achat et `PromoProximity` présente une corrélation presque nulle. Cela ne signifie pas nécessairement qu'elle est inutile pour le modèle . 

### Étape 4 — Matrice de corrélation entre les variables numériques

In [ ]:

# Sélectionner les variables numériques
numeric_cols = [
    'AdminPages', 'AdminTimeSec',
    'InfoPages', 'InfoTimeSec',
    'CatalogPages', 'CatalogTimeSec',
    'BounceRate', 'ExitRate',
    'EngagementValue', 'PromoProximity'
]

# Calculer la matrice de corrélation
corr_matrix = df_train[numeric_cols].corr()

# Afficher la matrice sous forme de heatmap
plt.figure(figsize=(12, 9))

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1
)

plt.title("Matrice de corrélation entre les variables numériques")
plt.tight_layout()
plt.show()


### Interprétation de la matrice de corrélation

L'analyse de la matrice de corrélation révèle plusieurs relations importantes entre les variables numériques.

On observe quatre fortes corrélations positives :

- `AdminPages` et `AdminTimeSec` : 0,74.

Interprétation :

Il existe une forte relation positive entre le nombre de pages administratives consultées et le temps passé sur ces pages.

Cela signifie que les visiteurs qui consultent davantage de pages administratives ont généralement tendance à y passer plus de temps.

- `InfoPages` et `InfoTimeSec` : 0,71.

Interprétation :

Il existe une forte relation positive entre le nombre de pages informatives consultées et le temps passé sur ces pages.

Les visiteurs qui consultent davantage de pages informatives ont généralement tendance à y passer plus de temps.

- `CatalogPages` et `CatalogTimeSec` : 0,92.

Interprétation :

C'est la relation la plus forte de toute la matrice.

Les visiteurs qui consultent davantage de pages du catalogue passent généralement plus de temps à les consulter.

Ces deux variables présentent donc une forte similarité dans les informations qu'elles apportent.

- `BounceRate` et `ExitRate` : 0,90.

Interprétation :

Il existe une très forte relation positive entre le taux de rebond et le taux de sortie.

Lorsque le taux de rebond augmente, le taux de sortie a également tendance à augmenter.

Ces deux variables peuvent donc contenir des informations similaires.

La relation la plus forte concerne `CatalogPages` et `CatalogTimeSec`, suivies de `BounceRate` et `ExitRate`.

Ces résultats indiquent que certaines variables peuvent contenir des informations similaires, ce qui peut entraîner une certaine redondance (`Deux variables fortement corrélées peuvent apporter des informations similaires au modèle`) , cela ne signifie pas qu'il faut automatiquement supprimer une variable.

Les autres variables présentent généralement des corrélations faibles ou modérées entre elles. En particulier, `PromoProximity` présente des corrélations presque nulles avec les autres variables.